# ImageCLEF 2026 — Submission Pipeline

Clean notebook for generating competition submissions.  
Assumes data + fine-tuned adapter already saved in vault from previous run.

In [ ]:
!pip install -q tf-keras
!pip install -q --user datasets transformers>=4.57.0 accelerate peft bitsandbytes pillow qwen-vl-utils

In [ ]:
import torch, json, os, time, re, gc, random
from collections import Counter, defaultdict
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm

print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.0f} GB")

# ── Paths ──────────────────────────────────────────────────────
VAULT_DIR = Path("/home/jovyan/vault/CLEF/ImageCLEF")
DATA_DIR = VAULT_DIR / "data"
OUTPUT_DIR = VAULT_DIR / "outputs"
ADAPTER_PATH = OUTPUT_DIR / "qlora_Qwen3-VL-8B-Thinking" / "final_adapter"
MODEL_ID = "Qwen/Qwen3-VL-8B-Thinking"

os.environ["HF_HOME"] = str(VAULT_DIR / "hf_cache")

print(f"Data: {DATA_DIR}")
print(f"Adapter: {ADAPTER_PATH}")
print(f"Adapter exists: {ADAPTER_PATH.exists()}")

## 1. Load Fine-Tuned Model

In [ ]:
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from peft import PeftModel

print("Loading base model + fine-tuned adapter...")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID, device_map="auto", torch_dtype=torch.bfloat16,
    trust_remote_code=True, attn_implementation="sdpa",
)
model = PeftModel.from_pretrained(base_model, str(ADAPTER_PATH))
model.eval()
print(f"Model loaded. GPU: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 2. Inference Functions

In [ ]:
# ── Prompts ────────────────────────────────────────────────────
THINKING_PROMPT = (
    "Look at this exam question image carefully.\n"
    "1. Read the question and ALL answer options.\n"
    "2. If there are diagrams, graphs, tables, or visual elements, analyze them.\n"
    "3. Think step by step about the correct answer.\n"
    "4. Select exactly ONE answer.\n\n"
    "Answer with a single letter: A, B, C, D, or E."
)

OPENQA_PROMPT = (
    "Look at this exam question image carefully.\n"
    "Read the question and analyze any visual elements.\n"
    "Provide a concise, accurate answer. Be brief.\n\n"
    "Answer:"
)

LANG_PROMPTS = {
    "English": "Answer with a single letter: A, B, C, D, or E.",
    "Chinese": "用一个字母回答：A、B、C、D 或 E。",
    "Bulgarian": "Отговорете само с една буква: A, B, C, D или E.",
    "Croatian": "Odgovorite samo jednim slovom: A, B, C, D ili E.",
    "Serbian": "Одговорите само једним словом: A, B, C, D или E.",
    "Italian": "Rispondi con una sola lettera: A, B, C, D o E.",
}


def build_prompt(base_prompt, subject="", language=""):
    prefix = f"You are an expert in {subject}. " if subject and subject != "unknown" else ""
    prompt = prefix + base_prompt
    if language in LANG_PROMPTS and language != "English":
        prompt = prompt.replace(
            "Answer with a single letter: A, B, C, D, or E.",
            LANG_PROMPTS[language]
        )
    return prompt


# ── Answer Extraction (handles thinking model) ────────────────
def extract_answer(response):
    if "</think>" in response:
        after = response.split("</think>")[-1].strip()
        after = re.sub(r'<\|[^>]+\|>', '', after).strip()
        if after and after[0].upper() in {"A", "B", "C", "D", "E"}:
            return after[0].upper()
        before = response.split("</think>")[0]
        last_lines = " ".join(before.strip().split("\n")[-3:])
        for pattern in [
            r'(?:answer|option|correct)\s*(?:is|:)?\s*\**\s*([A-E])\b',
            r'\b([A-E])\s*[\.)\s]*$',
            r'\*\*([A-E])\*\*',
            r'(?:选|答案|选项)\s*[:：]?\s*([A-E])',
        ]:
            match = re.search(pattern, last_lines, re.IGNORECASE)
            if match:
                return match.group(1).upper()
        for ch in reversed(last_lines):
            if ch.upper() in {"A", "B", "C", "D", "E"}:
                return ch.upper()
    clean = re.sub(r'<\|[^>]+\|>', '', response).strip()
    for pattern in [
        r'(?:answer|Answer|ANSWER)\s*(?:is|:)?\s*([A-E])',
        r'\*\*([A-E])\*\*',
        r'(?:选|答案)\s*[:：]?\s*([A-E])',
    ]:
        match = re.search(pattern, clean)
        if match:
            return match.group(1).upper()
    for ch in reversed(clean):
        if ch.upper() in {"A", "B", "C", "D", "E"}:
            return ch.upper()
    return "A"


# ── MCQ Prediction ─────────────────────────────────────────────
def predict_mcq(image_path, prompt_text):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": f"file://{image_path}"},
        {"type": "text", "text": prompt_text},
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(
        text=[text], images=[Image.open(image_path).convert("RGB")],
        return_tensors="pt", padding=True,
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=1024, do_sample=False)
    resp = processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False).strip()
    return extract_answer(resp), resp


print("Functions ready.")

## 3. Quick Sanity Check (5 samples)

In [ ]:
with open(DATA_DIR / "exams_v" / "validation_metadata.json") as f:
    val_data = json.load(f)

correct = 0
for item in val_data[:5]:
    img = str(DATA_DIR / "exams_v" / "validation" / "images" / f"{item['id']}.png")
    p = build_prompt(THINKING_PROMPT, item.get("subject", ""), item.get("language", ""))
    ans, resp = predict_mcq(img, p)
    ok = ans == item["answer_key"]
    if ok: correct += 1
    has_think = "</think>" in resp
    print(f"Gold: {item['answer_key']} | Pred: {ans} | {'OK' if ok else 'WRONG'} | </think>: {has_think}")

print(f"\nSanity check: {correct}/5")
print("If </think> appears in most → 1024 tokens is enough. Proceed to submissions.")

## 4. Visual MCQ Test (~9h)

1,117 test samples. No answer keys — submit to leaderboard.

In [ ]:
def run_mcq_test(dataset_name, experiment_name):
    """Run MCQ inference on competition test data."""
    meta_path = DATA_DIR / dataset_name / "test_metadata.json"
    image_dir = DATA_DIR / dataset_name / "test" / "images"

    with open(meta_path) as f:
        data = json.load(f)

    exp_dir = OUTPUT_DIR / experiment_name
    exp_dir.mkdir(parents=True, exist_ok=True)
    results_path = exp_dir / "predictions.json"

    # Resume from checkpoint
    results = []
    done_ids = set()
    if results_path.exists():
        with open(results_path) as f:
            results = json.load(f)
        done_ids = {r["question_id"] for r in results}
        print(f"  Resuming: {len(done_ids)} already done")

    remaining = [it for it in data if it["question_id"] not in done_ids]
    print(f"  {dataset_name}: {len(remaining)} remaining of {len(data)}")

    t0 = time.time()
    for i, item in enumerate(tqdm(remaining, desc=experiment_name)):
        img_path = str(image_dir / f"{item['question_id']}.png")
        if not os.path.exists(img_path):
            continue

        prompt = build_prompt(THINKING_PROMPT, item.get("subject", ""), item.get("language", ""))
        answer, _ = predict_mcq(img_path, prompt)

        results.append({
            "question_id": item["question_id"],
            "answer_key": answer,
            "language": item.get("language", ""),
            "subject": item.get("subject", ""),
        })

        # Checkpoint every 50
        if (i + 1) % 50 == 0:
            with open(results_path, "w") as f:
                json.dump(results, f, indent=2)
            elapsed = time.time() - t0
            rate = (i + 1) / elapsed
            eta = (len(remaining) - i - 1) / rate / 3600
            print(f"  [{i+1}/{len(remaining)}] {rate:.2f}/s ETA: {eta:.1f}h")

    with open(results_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"  Done: {len(results)} predictions saved")
    return results


print("="*55)
print("VISUAL MCQ TEST")
print("="*55)
visual_results = run_mcq_test("mcq_visual", "mcq_visual_final")

## 5. Text MCQ Test (~14h)

In [ ]:
print("="*55)
print("TEXT MCQ TEST")
print("="*55)
text_results = run_mcq_test("mcq_textual", "mcq_textual_final")

## 6. OpenQA Tests (~7h)

Run only if time remains after MCQ.

In [ ]:
def run_openqa_test(dataset_name, experiment_name):
    """Run OpenQA inference on competition test data."""
    meta_path = DATA_DIR / dataset_name / "test_metadata.json"
    image_dir = DATA_DIR / dataset_name / "test" / "images"

    with open(meta_path) as f:
        data = json.load(f)

    exp_dir = OUTPUT_DIR / experiment_name
    exp_dir.mkdir(parents=True, exist_ok=True)
    results_path = exp_dir / "predictions.json"

    results = []
    done_ids = set()
    if results_path.exists():
        with open(results_path) as f:
            results = json.load(f)
        done_ids = {r["question_id"] for r in results}
        print(f"  Resuming: {len(done_ids)} already done")

    remaining = [it for it in data if it["question_id"] not in done_ids]
    print(f"  {dataset_name}: {len(remaining)} remaining of {len(data)}")

    t0 = time.time()
    for i, item in enumerate(tqdm(remaining, desc=experiment_name)):
        img_path = str(image_dir / f"{item['question_id']}.png")
        if not os.path.exists(img_path):
            results.append({"question_id": item["question_id"], "answers": [""], "language": item.get("language", "")})
            continue

        prompt = build_prompt(OPENQA_PROMPT, item.get("subject", ""), item.get("language", ""))
        messages = [{"role": "user", "content": [
            {"type": "image", "image": f"file://{img_path}"},
            {"type": "text", "text": prompt},
        ]}]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(
            text=[text], images=[Image.open(img_path).convert("RGB")],
            return_tensors="pt", padding=True,
        ).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=512, do_sample=False)
        resp = processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

        results.append({"question_id": item["question_id"], "answers": [resp], "language": item.get("language", "")})

        if (i + 1) % 50 == 0:
            with open(results_path, "w") as f:
                json.dump(results, f, indent=2)

    with open(results_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"  Done: {len(results)} predictions saved")
    return results


print("="*55)
print("OPENQA VISUAL TEST")
print("="*55)
oqa_vis = run_openqa_test("openqa_visual", "openqa_visual_final")

print("\n" + "="*55)
print("OPENQA TEXTUAL TEST")
print("="*55)
oqa_txt = run_openqa_test("openqa_textual", "openqa_textual_final")

## 7. Package Submissions

In [ ]:
SUB_DIR = OUTPUT_DIR / "submissions"
SUB_DIR.mkdir(parents=True, exist_ok=True)

def package_mcq(experiment_name, output_name):
    path = OUTPUT_DIR / experiment_name / "predictions.json"
    if not path.exists():
        print(f"  {output_name}: NOT FOUND")
        return
    with open(path) as f:
        preds = json.load(f)
    sub = [{"question_id": p["question_id"], "answer_key": p["answer_key"]} for p in preds]
    ids = set()
    for item in sub:
        assert item["answer_key"] in {"A","B","C","D","E"}, f"Bad answer: {item}"
        assert item["question_id"] not in ids, f"Duplicate: {item['question_id']}"
        ids.add(item["question_id"])
    out = SUB_DIR / f"{output_name}.json"
    with open(out, "w") as f:
        json.dump(sub, f, indent=2, ensure_ascii=False)
    print(f"  {output_name}: {len(sub)} predictions -> {out}")

def package_openqa(experiment_name, output_name):
    path = OUTPUT_DIR / experiment_name / "predictions.json"
    if not path.exists():
        print(f"  {output_name}: NOT FOUND")
        return
    with open(path) as f:
        preds = json.load(f)
    sub = [{"question_id": p["question_id"], "answers": p["answers"], "language": p.get("language","")} for p in preds]
    out = SUB_DIR / f"{output_name}.json"
    with open(out, "w") as f:
        json.dump(sub, f, indent=2, ensure_ascii=False)
    print(f"  {output_name}: {len(sub)} predictions -> {out}")

print("Packaging Submissions")
print("="*55)
package_mcq("mcq_visual_final", "visual_mcq_normal")
package_mcq("mcq_textual_final", "text_mcq_normal")
package_openqa("openqa_visual_final", "visual_openqa_normal")
package_openqa("openqa_textual_final", "text_openqa_normal")

print(f"\nSubmissions in: {SUB_DIR}")
print()
print("Upload to:")
print("  Visual MCQ:    https://ai4media-bench.aimultimedialab.ro/competitions/12/")
print("  Text MCQ:      https://ai4media-bench.aimultimedialab.ro/competitions/16/")
print("  OpenQA Visual: check AI4Media-Bench")
print("  OpenQA Text:   check AI4Media-Bench")